# Part 2

## Load Part 1 Data

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

netid = 'hsaleh5'
root = Path.cwd()
summary = pd.read_csv(root / f'{netid}_project_summary.csv', sep=';')
summary['date'] = pd.to_datetime(summary['time'], unit='s', utc=True)
summary.head()

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


,project_wocid,commit_sha1,author,time,commit message,date
0,vibansal_hapcut2,0096d0226fd452ccc406c14b00e6f23abfa111c4,Peter Edge <edge.peterj@gmail.com>,1475273660,Update hapcut2.c\n\nremoved --v as an option f...,2016-09-30 22:14:20+00:00
1,vibansal_hapcut2,00a663d51502f56bad02deb82d5f4e48a4ef8736,Peter Edge <edge.peterj@gmail.com>,1459566607,added scripts for new versions of known and un...,2016-04-02 03:10:07+00:00
2,vibansal_hapcut2,0121162b713abd259815c98943b04ab1d423d73f,Peter Edge <edge.peterj@gmail.com>,1493502480,Lifted limit on number of nodes in iterative l...,2017-04-29 21:48:00+00:00
3,vibansal_hapcut2,01af9003507d608df96b65bd65f7b6e9e8940284,Peter Edge <edge.peterj@gmail.com>,1462382028,fixed conflict\n,2016-05-04 17:13:48+00:00
4,vibansal_hapcut2,024e1b905a54cf0e68e4d05c45dea00aa905be71,Vikas Bansal <vikas115@gmail.com>,1555725262,working implementation of phased/unphased geno...,2019-04-20 01:54:22+00:00


## Create Monthly Commit Time Series

In [2]:
timeseries = {}
for project, group in summary.groupby('project_wocid'):
    months = group['date'].dt.tz_localize(None).dt.to_period('M')
    counts = months.value_counts().sort_index()
    full_months = pd.period_range(counts.index.min(), counts.index.max(), freq='M')
    timeseries[project] = pd.DataFrame({
        'Month': full_months.astype(str),
        '#Commits': counts.reindex(full_months, fill_value=0).values
    })

display(timeseries[next(iter(timeseries))].head())

,Month,#Commits
0,2017-02,2
1,2017-03,2
2,2017-04,14
3,2017-05,0
4,2017-06,0


## Save Monthly CSV Files

In [3]:
for project, data in timeseries.items():
    data.to_csv(root / f'{netid}_commits_timeseries_{project}.csv', sep=';', index=False)

print('Saved', len(timeseries), 'CSV files')

Saved 10 CSV files


## Plot Commit Activity

In [4]:
for project, data in timeseries.items():
    dates = pd.to_datetime(data['Month'])
    plt.figure(figsize=(10, 5))
    plt.plot(dates, data['#Commits'])
    plt.xlabel('Time (YYYY-MM)')
    plt.ylabel('Number of Commits')
    plt.title(project)
    plt.grid(True)
    plt.gca().xaxis.set_major_locator(mdates.YearLocator(2))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(root / f'{netid}_timeseries_{project}.png', dpi=300)
    plt.close()

print('Saved', len(timeseries), 'plots')

Saved 10 plots


## Calculate Inactivity Gaps

In [6]:
gap_rows = []
for project, data in timeseries.items():
    gaps = []
    start = None
    for index, commits in enumerate(data['#Commits']):
        if commits == 0 and start is None:
            start = index
        if commits > 0 and start is not None:
            gaps.append((start, index - 1))
            start = None
    if start is not None:
        gaps.append((start, len(data) - 1))
    if gaps:
        longest = max(gaps, key=lambda gap: gap[1] - gap[0] + 1)
        gap_start = data.loc[longest[0], 'Month']
        gap_end = data.loc[longest[1], 'Month']
        gap_length = longest[1] - longest[0] + 1
        commits_after = int(data.loc[longest[1] + 1:, '#Commits'].sum())
    else:
        gap_start = 'N/A'
        gap_end = 'N/A'
        gap_length = 0
        commits_after = 0
    gap_rows.append({
        'Project': project,
        'LongestGapStart': gap_start,
        'LongestGapEnd': gap_end,
        'LongestGapLength': gap_length,
        'NumCommitsAfterLastGap': commits_after,
        'NumberOfGapsInTimeline': sum(end - begin + 1 >= 3 for begin, end in gaps)
    })

gap_stats = pd.DataFrame(gap_rows)
display(gap_stats)

,Project,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
0,antvis_g6,2017-05,2017-06,2,16337,0
1,dsavransky_exosims,2016-11,2016-12,2,4215,0
2,elki-project_elki,2023-11,2024-01,3,221,2
3,friendly_heplots,2007-06,2008-10,17,718,16
4,lsstdesc_ccl,N/A,N/A,0,0,0
5,modelica_modelica_linearsystems2,2022-03,2023-05,15,431,16
6,neuroml_neuroml2,2023-10,2024-03,6,45,5
7,ropensci_gsodr,2015-08,2015-12,5,2899,7
8,vibansal_hapcut2,2022-11,2024-07,21,5,8
9,wangbolun300_fast-envelope,2024-02,2025-02,13,17,5


## Classify Activity Patterns

In [7]:
patterns = {}
for project, data in timeseries.items():
    third = len(data) // 3
    parts = [data.iloc[:third], data.iloc[third:third * 2], data.iloc[third * 2:]]
    first, middle, last = [part['#Commits'].mean() for part in parts]
    if middle < first * 0.65 and middle < last * 0.65:
        pattern = 'U-shaped'
    elif last > first * 1.35:
        pattern = 'rising'
    elif last < first * 0.65:
        pattern = 'declining'
    elif data['#Commits'].std() <= data['#Commits'].mean():
        pattern = 'steady'
    else:
        pattern = 'irregular'
    patterns[project] = pattern

patterns

{'antvis_g6': 'irregular',
 'dsavransky_exosims': 'irregular',
 'elki-project_elki': 'declining',
 'friendly_heplots': 'rising',
 'lsstdesc_ccl': 'declining',
 'modelica_modelica_linearsystems2': 'rising',
 'neuroml_neuroml2': 'U-shaped',
 'ropensci_gsodr': 'declining',
 'vibansal_hapcut2': 'declining',
 'wangbolun300_fast-envelope': 'declining'}

## Create Project Statistics CSV

In [8]:
part1_stats = pd.read_csv(root / f'{netid}_project_stats.csv', sep=';')
part2_columns = ['LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'NumberOfGapsInTimeline', 'ActivityPattern']
stats = part1_stats.drop(columns=part2_columns, errors='ignore').merge(gap_stats, on='Project', how='left')
stats['ActivityPattern'] = stats['Project'].map(patterns)
stats.to_csv(root / f'{netid}_project_stats.csv', sep=';', index=False)
display(stats)

,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,vibansal_hapcut2,588,22,1405548177,1741933800,233,39,2025-02-09,2022-11,2024-07,21,5,8,declining
1,friendly_heplots,721,8,1173790214,1757692802,11,5,2026-09-13,2007-06,2008-10,17,718,16,rising
2,elki-project_elki,15704,94,1121867691,1775749738,831,321,2026-08-10,2023-11,2024-01,3,221,2,declining
3,neuroml_neuroml2,986,18,1352295427,1739353578,66,26,2026-09-02,2023-10,2024-03,6,45,5,U-shaped
4,ropensci_gsodr,2912,18,1427153177,1762248461,98,19,2026-05-17,2015-08,2015-12,5,2899,7,declining
5,modelica_modelica_linearsystems2,1501,22,1251871200,1761568752,26,21,2026-09-24,2022-03,2023-05,15,431,16,rising
6,wangbolun300_fast-envelope,488,13,1559158533,1751287292,64,14,2025-06-30,2024-02,2025-02,13,17,5,declining
7,antvis_g6,16355,474,1486980044,1762258188,12318,1646,2026-09-23,2017-05,2017-06,2,16337,0,irregular
8,lsstdesc_ccl,8694,190,1461938918,1762533598,160,87,2026-07-28,N/A,N/A,0,0,0,declining
9,dsavransky_exosims,4319,66,1433177497,1762203232,36,49,2026-09-29,2016-11,2016-12,2,4215,0,irregular


## Interpretation

- vibansal_hapcut2 is declining because activity is lower in the later part of the timeline. Its longest gap was 21 months from 2022-11 to 2024-07, and it has 8 gaps of at least 3 months.
- friendly_heplots is rising because the later years have more activity than the early years. Its longest gap was 17 months from 2007-06 to 2008-10, and it has 16 gaps of at least 3 months.
- elki-project_elki is declining because its later monthly average is lower. Its longest gap was 3 months from 2023-11 to 2024-01, and it has 2 gaps of at least 3 months.
- neuroml_neuroml2 is U-shaped because activity dropped in the middle and increased again later. Its longest gap was 6 months from 2023-10 to 2024-03, and it has 5 gaps of at least 3 months.
- ropensci_gsodr is declining because recent activity is lower than its earlier activity. Its longest gap was 5 months from 2015-08 to 2015-12, and it has 7 gaps of at least 3 months.
- modelica_modelica_linearsystems2 is rising because the later part has more commits. Its longest gap was 15 months from 2022-03 to 2023-05, and it has 16 gaps of at least 3 months.
- wangbolun300_fast-envelope is declining because its activity falls in the later part. Its longest gap was 13 months from 2024-02 to 2025-02, and it has 5 gaps of at least 3 months.
- antvis_g6 is irregular because commit counts change a lot between months. Its longest gap was 2 months from 2017-05 to 2017-06, and it has 0 gaps of at least 3 months.
- lsstdesc_ccl is declining because its later monthly average is lower, but it still has commits every month. It has no inactivity gap and 0 gaps of at least 3 months.
- dsavransky_exosims is irregular because activity changes without a clear long-term direction. Its longest gap was 2 months from 2016-11 to 2016-12, and it has 0 gaps of at least 3 months.

# Parts 3 and 4

I used the gaps from Part 2 and kept the original commit records, including merges and commits from other branches. The before/after samples are sorted by timestamp, with SHA as the tie breaker. G6 and EXOSIMS have two-month gaps, so I still examine them even though they have no gaps of at least three months. CCL has no gap, so it has no pre/post rows or gap themes.

Status and recent themes use the last 10 GitHub default-branch commits on or before **2025-09-30**, as requested. Active means the latest of those commits is on or after 2025-04-01. Part 1's `lastGHCommitDate` stays as originally collected; it is a later snapshot. GitHub uses committer dates here, while the original CSV uses author timestamps, so the two timelines can differ.

## Commits around the longest gap

In [9]:
stats = pd.read_csv(root / f'{netid}_project_stats.csv', sep=';', keep_default_na=False)
columns = ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
gap_samples = {}
gap_rows = []
for _, row in stats.iterrows():
    project = row['Project']
    data = summary[summary['project_wocid'] == project].sort_values(['date', 'commit_sha1'])
    if row['LongestGapLength'] == 0:
        gap_samples[project] = {'pre': data.iloc[:0], 'post': data.iloc[:0]}
        continue
    start = pd.Timestamp(row['LongestGapStart'], tz='UTC')
    end = pd.Timestamp(row['LongestGapEnd'], tz='UTC') + pd.offsets.MonthBegin(1)
    samples = {'pre': data[data['date'] < start].tail(10),
               'post': data[data['date'] >= end].head(10)}
    gap_samples[project] = samples
    for side, sample in samples.items():
        output = sample[columns].copy()
        output.insert(0, 'pre/post', side)
        gap_rows.append(output)

gap_commits = pd.concat(gap_rows, ignore_index=True)
gap_commits.to_csv(root / f'{netid}_project_gap_commits.csv', sep=';', index=False)
display(gap_commits.groupby(['project_wocid', 'pre/post']).size().unstack(fill_value=0))
print('CCL has no gap, so no pre/post rows were added for it.')

pre/post,post,pre
project_wocid,,
antvis_g6,10,10
dsavransky_exosims,10,10
elki-project_elki,10,10
friendly_heplots,10,3
modelica_modelica_linearsystems2,10,10
neuroml_neuroml2,10,10
ropensci_gsodr,10,10
vibansal_hapcut2,5,10
wangbolun300_fast-envelope,10,10


CCL has no gap, so no pre/post rows were added for it.


## Recent GitHub commits

Each request retrieves 10 commits, with a one-second wait between projects. These are the latest commits at the assignment cutoff, rather than the first commits after a gap.

In [10]:
import json
import requests
import time

metadata = json.loads((root / f'{netid}_part1_metadata.json').read_text(encoding='utf-8'))
repos = {p['Project']: p['github_repo'] for p in metadata['projects']}
recent = {}
for project, repo in repos.items():
    response = requests.get(f'https://api.github.com/repos/{repo}/commits',
                            params={'until': '2025-09-30T23:59:59Z', 'per_page': 10},
                            timeout=30)
    response.raise_for_status()
    recent[project] = pd.DataFrame([
        {'sha': c['sha'], 'date': c['commit']['committer']['date'],
         'author': c['commit']['author']['name'], 'message': c['commit']['message']}
        for c in response.json()
    ])
    time.sleep(1)

recent_table = pd.concat([data.assign(Project=project) for project, data in recent.items()],
                         ignore_index=True)[['Project', 'sha', 'date', 'author', 'message']]
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100):
    display(recent_table)

,Project,sha,date,author,message
0,vibansal_hapcut2,9a10aba2a6782cd46327faf6a2271d41002c81b3,2025-02-09T04:18:32Z,vbansal,update extracthairs.c to parse cram files
1,vibansal_hapcut2,0eb7075424afbb40259ced6b5b1cbd95d0cedf55,2022-10-14T21:50:21Z,Vikas Bansal,more changes to readme
2,vibansal_hapcut2,0fa74155482e2b1539242beba8be3c98a23de00d,2022-10-14T21:44:51Z,Vikas Bansal,minor fix
3,vibansal_hapcut2,da74c8852c641ff2a6fa50ff393d17b96f69facd,2022-10-14T21:42:17Z,Vikas Bansal,minor changes to readme
4,vibansal_hapcut2,32b0547f4ca8061a9cd992e0748fe1e24abc4659,2022-05-05T18:18:45Z,Vikas Bansal,Merge branch 'snehring-feature/newer-gcc-compat'
5,vibansal_hapcut2,96f924e18d70add1841fe5b11c60e8d8d2a872b2,2022-04-27T16:34:33Z,snehring,Makefile changes
6,vibansal_hapcut2,65771565f29aa61f1fa7e8319a39c8a65cc19824,2022-04-27T16:28:35Z,snehring,fixing build errors with newer gcc versions
7,vibansal_hapcut2,66ee827f9130fa64ff93044227702804308f1650,2022-01-31T21:50:02Z,Vikas Bansal,updated installation instructions
8,vibansal_hapcut2,c8fa7a482f8422aa7e3c03bf5a483bf3f485b7f0,2022-01-31T21:46:05Z,Vikas Bansal,- added script install.sh to install htslib (from source) and hapcut2 automatically\n\n- path to...
9,vibansal_hapcut2,8f6ce5774f5eea08cdba994807f0e008f242d790,2021-06-15T23:24:24Z,Bansal Lab,Merge pull request #114 from pontushojer/linkfrag-update\n\nEnable the use of the MI tag to dete...


## Commit themes

I read the messages and assigned one main theme per commit. Tests, formatting, vague messages and plain merges go under Other. A merge with a clear description uses that description's theme. Ties use the category order below. The labels follow each displayed sample's order; recent commits run newest to oldest. A gap with no commits on either side gets N/A.

In [11]:
from collections import Counter

themes = {
    'F': 'Feature development', 'B': 'Bug fixes', 'D': 'Documentation updates',
    'P': 'Dependency updates', 'R': 'Release', 'S': 'Security patches',
    'A': 'Automated bot contributions', 'O': 'Other'
}
labels = {
    'vibansal_hapcut2': {'pre': 'F B B B D B D D D O', 'post': 'B O F O O',
                        'recent': 'F D D D B B B D F F'},
    'friendly_heplots': {'pre': 'D R F', 'post': 'R R R R R R F D D D',
                        'recent': 'D D B F R R R D B B'},
    'elki-project_elki': {'pre': 'D D D D D D F F D D', 'post': 'P P O O O O O O O O',
                         'recent': 'D F F O B O O O B D'},
    'neuroml_neuroml2': {'pre': 'P O R R D D D D D R', 'post': 'F F F B F F O P O F',
                        'recent': 'R D P R D D R O R O'},
    'ropensci_gsodr': {'pre': 'D D D D O D D F O D', 'post': 'B O B B F B P D F D',
                      'recent': 'R D R B R R D D R O'},
    'modelica_modelica_linearsystems2': {
        'pre': 'D D D D B D D O B O', 'post': 'O O D B B O O P P P',
        'recent': 'F O D O F B B D O D'},
    'wangbolun300_fast-envelope': {
        'pre': 'P B O B P B D O P B', 'post': 'B B O O O P O O O P',
        'recent': 'D D O B O O O P O O'},
    'antvis_g6': {'pre': 'O O D D D D D D O O', 'post': 'R R R R R R D D R B',
                  'recent': 'P P O D B F D D D B'},
    'lsstdesc_ccl': {'pre': '', 'post': '', 'recent': 'F F D F B B B B D O'},
    'dsavransky_exosims': {'pre': 'B B B D D D B B F O', 'post': 'B D O D B B O P O O',
                          'recent': 'A A A A B A B A O F'}
}

def top_two(codes):
    counts = Counter(codes.split())
    ordered = sorted(counts, key=lambda key: (-counts[key], list(themes).index(key)))
    return ':'.join(themes[key] for key in ordered[:2]) or 'N/A'

reviewed = []
for project, samples in gap_samples.items():
    for side, sample in samples.items():
        codes = labels[project][side].split()
        assert len(sample) == len(codes)
        for (_, commit), theme in zip(sample.iterrows(), codes):
            reviewed.append({'Project': project, 'sample': side, 'sha': commit['commit_sha1'],
                             'message': commit['commit message'], 'Theme': themes[theme]})
    assert len(recent[project]) == len(labels[project]['recent'].split())
    stats.loc[stats['Project'] == project, 'BeforeThemes'] = top_two(labels[project]['pre'])
    stats.loc[stats['Project'] == project, 'AfterThemes'] = top_two(labels[project]['post'])
    stats.loc[stats['Project'] == project, 'RecentThemes'] = top_two(labels[project]['recent'])

display(pd.DataFrame(reviewed))
display(stats[['Project', 'BeforeThemes', 'AfterThemes', 'RecentThemes']])

,Project,sample,sha,message,Theme
0,vibansal_hapcut2,pre,717c934c82fc6f66f44e1304f0700a206db78b9d,adding code for extracting haplotype-informati...,Feature development
1,vibansal_hapcut2,pre,65771565f29aa61f1fa7e8319a39c8a65cc19824,fixing build errors with newer gcc versions\n,Bug fixes
2,vibansal_hapcut2,pre,96f924e18d70add1841fe5b11c60e8d8d2a872b2,Makefile changes\n,Bug fixes
3,vibansal_hapcut2,pre,32b0547f4ca8061a9cd992e0748fe1e24abc4659,Merge branch 'snehring-feature/newer-gcc-compa...,Bug fixes
4,vibansal_hapcut2,pre,b2744f31a2b3ff2e5ee6f427ad2c5d84dfbdde05,"docs fix: add --ref for extractHAIRS, fix --ea...",Documentation updates
...,...,...,...,...,...
163,dsavransky_exosims,post,74d8b656a110fdc896762f7bc797235ba7c418f7,Bugfix\n\n- Runtime errors: divide-by-zero err...,Bug fixes
164,dsavransky_exosims,post,e789c6d0df692a18c87dc10534d1a4f960967dfe,adding jpl unit tests\n,Other
165,dsavransky_exosims,post,5e104e0e5f9546c20742e394b4c818c556373907,"Forecaster update\n\n- Now, user needs to down...",Dependency updates
166,dsavransky_exosims,post,1b6c34e33cdf78d7af83aedcf211e6868fafa6b3,adding EXOSIMS nonpublic defs to gitignore\n,Other


,Project,BeforeThemes,AfterThemes,RecentThemes
0,vibansal_hapcut2,Bug fixes:Documentation updates,Other:Feature development,Documentation updates:Feature development
1,friendly_heplots,Feature development:Documentation updates,Release:Documentation updates,Bug fixes:Documentation updates
2,elki-project_elki,Documentation updates:Feature development,Other:Dependency updates,Other:Feature development
3,neuroml_neuroml2,Documentation updates:Release,Feature development:Other,Release:Documentation updates
4,ropensci_gsodr,Documentation updates:Other,Bug fixes:Feature development,Release:Documentation updates
5,modelica_modelica_linearsystems2,Documentation updates:Bug fixes,Other:Dependency updates,Documentation updates:Other
6,wangbolun300_fast-envelope,Bug fixes:Dependency updates,Other:Bug fixes,Other:Documentation updates
7,antvis_g6,Documentation updates:Other,Release:Documentation updates,Documentation updates:Bug fixes
8,lsstdesc_ccl,N/A,N/A,Bug fixes:Feature development
9,dsavransky_exosims,Bug fixes:Documentation updates,Other:Bug fixes,Automated bot contributions:Bug fixes


## Gap explanations and recovery

These are my best guesses from the commits, not confirmed explanations of maintainers' personal circumstances. I also checked GitHub issues/PRs around each gap and README history in the existing GitHub mirrors. The evidence and links below record those checks. Matching author emails helps identify returning people, but changed names/emails do not always mean a new contributor. Recovery means commits restarted after this gap; it is separate from status at the cutoff.

In [12]:
reviews = {'vibansal_hapcut2': {'reason': 'Likely sparse maintenance after the October 2022 documentation/build fixes, '
                                'rather than a confirmed shutdown. Issue #127 was opened on 2022-11-16 and '
                                'the maintainer replied only on 2024-08-21: '
                                'https://github.com/vibansal/HapCUT2/issues/127',
                      'recovered': 'Yes, limited; CSV activity resumed in August 2024, but default-branch '
                                   'work resumed in February 2025.',
                      'why': 'J.Chen submitted Makefile changes on 2024-08-13 (d71aa89101, PR #137, still '
                             'unmerged). The maintainer added CRAM parsing on 2025-02-09 (9a10aba2a6): '
                             'https://github.com/vibansal/HapCUT2/pull/137',
                      'who': 'New recorded contributor J.Chen made the first post-gap change; returning '
                             'maintainer Vikas Bansal/vbansal made the CRAM update. Simon Blanchoud also '
                             'appears in a post-gap merge (7681a0fc7c).',
                      'notes': 'There are 8 gaps of at least three months, and the longest lasted 21 months. '
                               'The first apparent recovery is an unmerged Makefile PR, so it overstates '
                               'recovery on the main branch. The last default-branch commit by the cutoff '
                               'was 2025-02-09, making the project Inactive.',
                      'reflection': 'The long pause is visible, but the reason is harder to prove. Before '
                                    'it, the commits mostly deal with builds and documentation. Issue #127 '
                                    'kept getting user reports before a maintainer response in August 2024, '
                                    'which suggests limited maintenance capacity, but does not explain the '
                                    "maintainer's circumstances. PR #137 is important because its Makefile "
                                    'changes appear in the CSV without being merged. The later CRAM-parsing '
                                    'change is stronger evidence that the original maintainer returned, '
                                    'although only briefly.',
                      'check': 'The October 2022 README changes explain command usage, with no shutdown '
                               'notice. PR #137 received a maintainer response on 2024-08-15 saying its '
                               'paths were local; it was not merged.',
                      'sources': [('User report and delayed reply',
                                   'https://github.com/vibansal/HapCUT2/issues/127'),
                                  ('Unmerged Makefile PR', 'https://github.com/vibansal/HapCUT2/pull/137')]},
 'friendly_heplots': {'reason': 'Likely a pause between early package versions or incomplete imported '
                                'history. Only 3 pre-gap commits exist, and c3ade355d1 (2007-05-23) has a '
                                'git-svn marker; no issue/PR records were found around this 2007-2008 gap.',
                      'recovered': 'Yes; commits resumed on 2008-11-04 (f9c5f2e2af).',
                      'why': 'Work resumed with version 0.8-2 updates (f9c5f2e2af, 2008-11-04), plotting '
                             'changes, and documentation. Commit 695aba26a3 (2009-01-26) records Michael '
                             'Friendly becoming maintainer.',
                      'who': 'friendly authored all 10 sampled post-gap commits and is new as a recorded '
                             "author. However, c3ade355d1 already credits Mike's changes before the gap, so "
                             'this need not mean a new person joined.',
                      'notes': 'There are 16 gaps of at least three months, with a 17-month longest gap '
                               'early in the history. Imported SVN records and only three earlier commits '
                               'make that gap hard to interpret. Later releases and a 2025-09-12 '
                               'default-branch commit support Active status at the cutoff.',
                      'reflection': 'This looks more like uneven early package work than permanent '
                                    'abandonment. The three commits before the gap include an initial '
                                    'package import and plotting changes, while the restart contains a batch '
                                    'of version 0.8-2 updates. The SVN markers make it hard to know how '
                                    'complete this early history is. Michael Friendly dominates the restart '
                                    'and later becomes maintainer, but was already credited in the earlier '
                                    'work. More recent bug fixes, documentation and releases fit the rising '
                                    'activity pattern.',
                      'check': 'No GitHub issues or PRs were found dated 2007-05 through 2008-11. The '
                               'sampled old trees have no README file, so there is no README evidence '
                               'explaining the pause.',
                      'sources': [('Early plotting change',
                                   'https://github.com/friendly/heplots/commit/c3ade355d1'),
                                  ('Maintainer update',
                                   'https://github.com/friendly/heplots/commit/695aba26a3')]},
 'elki-project_elki': {'reason': 'Possibly a short pause after an algorithm/documentation batch, not proven '
                                 'abandonment. The last pre-gap commit a8eda51c06 (author date 2023-10-30) '
                                 'documents Shallot; the first return 6e50897b79 (2024-02-17) reduces '
                                 'FastMath/JaFaMa usage.',
                       'recovered': 'Yes; author-dated commits resumed on 2024-02-17.',
                       'why': 'Erich Schubert resumed dependency-related changes and code cleanup '
                              '(6e50897b79, 2024-02-17). No issue/PR records were found around the gap, and '
                              'the README does not announce a shutdown or revival.',
                       'who': 'Erich Schubert authored all 10 sampled post-gap commits and already had '
                              'pre-gap commits. The immediate pre-gap sample is Andreas Lang, so the sampled '
                              'author changes without a new maintainer being established.',
                       'notes': 'There are 2 gaps of at least three months, and the longest is only three '
                                'months. Repeated messages under different SHAs and differing '
                                'author/committer dates make the gap less straightforward than the plot '
                                'suggests. The last default-branch commit by the cutoff was 2025-06-02, so '
                                'it was Active.',
                       'reflection': "The three-month pause is small compared with this project's long "
                                     'history. Documentation and algorithm work come before it, followed by '
                                     'dependency changes and code cleanup from an existing contributor. That '
                                     'suggests ordinary intermittent maintenance, although there is no '
                                     'direct explanation. Several messages appear under multiple SHAs, and '
                                     'a8eda51c06 has a 2023 author date but a 2025 GitHub committer date. I '
                                     'would be careful about treating this author-date gap as proof that all '
                                     'GitHub development stopped.',
                       'check': 'No GitHub issues or PRs were found dated 2023-10 through 2024-02. The '
                                'README at the nearest pre/post commits describes the research software and '
                                'has no abandonment or revival notice.',
                       'sources': [('Pre-gap documentation',
                                    'https://github.com/elki-project/elki/commit/a8eda51c06'),
                                   ("Returning contributor's maintenance",
                                    'https://github.com/elki-project/elki/commit/6e50897b79')]},
 'neuroml_neuroml2': {'reason': 'Likely a lull after release/schema work: 4af9426acc (2023-09-22) prepares '
                                'v1.10.0 with the NMLv2.3.1 schema. No issue/PR records were found during '
                                'October 2023-March 2024; there is no explicit reason for the pause.',
                      'recovered': 'Yes; work resumed in April 2024, but recovery did not ensure later '
                                   'activity at the cutoff.',
                      'why': 'Issue #220 requested explicit spikeThresh validation on 2024-04-11. Ankur '
                             'Sinha implemented it in 225e953f4b on 2024-04-17, and PR #222 was merged on '
                             '2024-04-18: https://github.com/NeuroML/NeuroML2/pull/222',
                      'who': 'Existing contributors Ankur Sinha and Padraig Gleeson/pgleeson authored the '
                             'first 10 post-gap commits; their emails also occur before the gap. PR #222 was '
                             'authored by sanjayankur31 and merged by pgleeson.',
                      'notes': 'There are 5 gaps of at least three months, with a six-month longest gap '
                               'after release work. The spikeThresh issue and PR clearly explain the '
                               'restart, but not why work paused. GitHub default-branch activity last '
                               'reached 2024-08-20 by the cutoff, so Currentstatus is Inactive even though '
                               'the CSV includes later branch commits.',
                      'reflection': 'The gap follows a release and schema update, so a quiet period after '
                                    'that work is plausible. I could not find a stated reason for the pause. '
                                    'The restart is much easier to explain: issue #220 identifies a missing '
                                    'validation requirement and PR #222 implements it a week later. The same '
                                    'contributors then continue schema, example and CI work. This is a '
                                    'recovery from the longest gap, but the default branch was still '
                                    "inactive under the assignment's six-month rule.",
                      'check': 'Issue #220 was opened on 2024-04-11 and PR #222 merged on 2024-04-18. The '
                               'pre/post README has no abandonment notice; pre-gap PR #219 updates the '
                               'documentation link.',
                      'sources': [('Validation request', 'https://github.com/NeuroML/NeuroML2/issues/220'),
                                  ('Implementation and merge',
                                   'https://github.com/NeuroML/NeuroML2/pull/222'),
                                  ('Pre-gap documentation update',
                                   'https://github.com/NeuroML/NeuroML2/pull/219')]},
 'ropensci_gsodr': {'reason': 'Likely intermittent work on a young package after documentation/citation '
                              'setup (e00db87ed9, 2015-07-13). The README stayed unchanged across the '
                              'nearest boundary commits, and no issues/PRs were found around the gap; a '
                              'specific cause is unknown.',
                    'recovered': 'Yes; commits resumed on 2016-01-27 (def440c2d4).',
                    'why': 'The restart fixes FTP download URLs and object names (def440c2d4 and e19ca0b20b, '
                           '2016-01-27), followed by a version 2.0 functionality merge (57be2151f9, '
                           '2016-03-29). These support renewed maintenance needs, not a confirmed personal '
                           'reason.',
                    'who': 'Adam H. Sparks authored all 10 sampled post-gap commits and the pre-gap work. '
                           'His email matches despite small name variations, so there is no maintainer '
                           'change in this sample.',
                    'notes': 'There are 7 gaps of at least three months, with a five-month longest gap near '
                             'the start. Download fixes and later feature work suggest the original '
                             'maintainer returned as maintenance needs arose. Recent CRAN release '
                             'preparation and a 2025-07-31 default-branch commit support Active status.',
                    'reflection': 'The five-month pause comes very early, after README and citation work. '
                                  'There are no matching GitHub issues or PRs explaining it, so an '
                                  'occasional development schedule is only a guess. The restart is clearer: '
                                  'FTP fixes and naming corrections are followed by broader weather-data '
                                  'selection features. Adam Sparks handles both sides, which makes this a '
                                  'return by the same maintainer. The later decline in commit counts does '
                                  'not mean abandonment, since CRAN submission work was still happening in '
                                  'July 2025.',
                    'check': 'No GitHub issues or PRs were found dated 2015-07 through 2016-01. The README '
                             'at e00db87ed9 and def440c2d4 describes installation and weather data, with no '
                             'shutdown announcement.',
                    'sources': [('Citation setup', 'https://github.com/ropensci/GSODR/commit/e00db87ed9'),
                                ('FTP correction', 'https://github.com/ropensci/GSODR/commit/e19ca0b20b'),
                                ('Version 2.0 functionality',
                                 'https://github.com/ropensci/GSODR/commit/57be2151f9')]},
 'modelica_modelica_linearsystems2': {'reason': 'Likely a maintenance lull after the February 2022 '
                                                'documentation and cleanup work (a69999dce5, 2022-02-08). No '
                                                'issues/PRs were found created during the 15-month gap, and '
                                                'the README has no shutdown notice; the actual cause is '
                                                'uncertain.',
                                      'recovered': 'Yes; work resumed on 2023-06-15 (cf486a8404).',
                                      'why': 'PR #155 removed a byte-order mark on 2023-06-15 and PR #156 '
                                             'improved descriptions on 2023-06-22. Later August crash fixes '
                                             'and September LAPACK changes show maintenance continuing: '
                                             'https://github.com/modelica/Modelica_LinearSystems2/pull/155',
                                      'who': 'Returning contributors Jakub Tobolar (tobolar) and Hans Olsson '
                                             '(HOS) authored the first 10 post-gap commits. Both have '
                                             'matching emails in the pre-gap history; Tobolar opened and '
                                             'merged PRs #155 and #156.',
                                      'notes': 'There are 16 gaps of at least three months, and the longest '
                                               'lasted 15 months. Small cleanup/documentation PRs restarted '
                                               'work under existing contributors, but they do not explain '
                                               'the original pause. The last default-branch commit by the '
                                               'cutoff was 2025-06-25, so it was Active.',
                                      'reflection': 'The many gaps make development look intermittent even '
                                                    'though the overall pattern is rising. Before the '
                                                    'longest pause, the work is mostly documentation and '
                                                    'small corrections. The return starts with encoding '
                                                    'cleanup and better descriptions, then moves into crash '
                                                    'fixes and library compatibility. The same contributors '
                                                    'are involved, so there is no clear takeover. The '
                                                    'restart is easy to identify from the PRs, but the '
                                                    'reason for the long pause remains uncertain.',
                                      'check': 'PR #155 was merged on 2023-06-15 and #156 on 2023-06-22. The '
                                               'pre/post README continues to describe the library; no '
                                               'archive or revival notice was found.',
                                      'sources': [('Encoding cleanup',
                                                   'https://github.com/modelica/Modelica_LinearSystems2/pull/155'),
                                                  ('Documentation PR',
                                                   'https://github.com/modelica/Modelica_LinearSystems2/pull/156')]},
 'wangbolun300_fast-envelope': {'reason': 'Likely occasional maintenance of an established research '
                                          'implementation: 38b2cfbbf0 (2024-01-11) makes a small template '
                                          'change before a 13-month pause. The README still cites the 2020 '
                                          'paper and does not announce abandonment; no issues/PRs were found '
                                          'during the gap itself.',
                                'recovered': 'Yes; a new contribution appeared on 2025-03-28, with PR #10 '
                                             'merged on 2025-04-06.',
                                'why': 'Daniel Zint submitted a compiler-warning fix in PR #10, merged by '
                                       'wangbolun300. PR #12 then updated packages and platform logic on '
                                       '2025-06-30, supporting a return for build compatibility: '
                                       'https://github.com/wangbolun300/fast-envelope/pull/10',
                                'who': 'Daniel Zint is new in the pre/post comparison and starts the '
                                       'recovery (d51d265d21). Existing maintainer Bolun Wang merges PR #10 '
                                       'and contributes later build work; Bolun Wang1 appears to be a '
                                       'name/email variant rather than a separate person.',
                                'notes': 'There are 5 gaps of at least three months, with a 13-month longest '
                                         "gap. A new contributor's build fix and the original maintainer's "
                                         'package/platform updates explain the restart better than the '
                                         'pause. The latest default-branch commit by the cutoff was '
                                         '2025-06-30, making it Active.',
                                'reflection': 'Activity drops after the initial research implementation, and '
                                              'the longest gap follows small dependency and template '
                                              'changes. A stable codebase needing only occasional attention '
                                              'is plausible, but the README does not give a direct '
                                              "explanation. Daniel Zint's warning fix provides a clear "
                                              'trigger for renewed contributions, and Bolun Wang merges it '
                                              'before updating build support. This is a mix of a new '
                                              'contributor and a returning maintainer, rather than evidence '
                                              'of a complete transfer of ownership.',
                                'check': 'PR #10 was opened on 2025-03-28 and merged on 2025-04-06. PR #12 '
                                         'merged package/platform fixes on 2025-06-30; the pre/post README '
                                         'has no abandonment notice.',
                                'sources': [('First returning PR',
                                             'https://github.com/wangbolun300/fast-envelope/pull/10'),
                                            ('Package and platform fixes',
                                             'https://github.com/wangbolun300/fast-envelope/pull/12')]},
 'antvis_g6': {'reason': 'Likely a short interval between early releases: README corrections end the pre-gap '
                         'sample on 2017-04-11, then version 1.1.1 work resumes on 2017-07-20 (79dc5d8865). '
                         'User issues continued during May-June, so this two-month commit gap does not '
                         'establish abandonment.',
               'recovered': 'Yes; release work resumed in July 2017, followed by later releases.',
               'why': 'The first post-gap commits update version 1.1.1; later ones publish 1.2.0. Issue #15 '
                      'provides a 1.0 migration guide just before the gap, consistent with release-focused '
                      'early activity: https://github.com/antvis/G6/issues/15',
               'who': 'Yellowthrough and 有田 authored the immediate pre/post work and both already appear '
                      'before the gap. Their paired commits under different SHAs should not be mistaken for '
                      'separate independent features.',
               'notes': 'There are no gaps of at least three months; the longest is only two months. Early '
                        'version releases and continued user issues make that pause look temporary. The last '
                        'default-branch commit by the cutoff was 2025-09-15, so the project was Active.',
               'reflection': 'The longest gap is only two months, so I would not call this a discontinued '
                             'project. Early README corrections are followed by releases from the same '
                             'recorded contributors. The 1.0 migration guide and user questions during the '
                             'gap show that the commit chart alone misses some project activity. The later '
                             'irregular pattern reflects changing amounts of work, rather than a '
                             'demonstrated seasonal cycle. Recent documentation, bug fixes and dependency '
                             'upgrades also show continued maintenance.',
               'check': 'Issue #15 documents the 1.0 migration on 2017-04-12, and issue #23 requests '
                        'TypeScript support on 2017-07-12. The July README changes simplify the feature list '
                        'without a revival announcement; no PRs were found in the searched early window.',
               'sources': [('Migration guide', 'https://github.com/antvis/G6/issues/15'),
                           ('User request near restart', 'https://github.com/antvis/G6/issues/23'),
                           ('Version restart', 'https://github.com/antvis/G6/commit/79dc5d8865')]},
 'lsstdesc_ccl': {'reason': 'N/A active Project',
                  'recovered': 'N/A; no zero-commit month in the observed timeline.',
                  'why': 'N/A; there is no inactivity gap to recover from. Feature and maintenance work '
                         'continues, including PR #1254 on 2025-09-29: '
                         'https://github.com/LSSTDESC/CCL/pull/1254',
                  'who': 'N/A; no recovery event. Recent work includes Elisa Chisari and Andrina Nicola on '
                         'features and Robert Reischke and Carlos Garcia-Garcia on fixes (PRs #1254, #1249, '
                         '#1248 and #1244).',
                  'notes': 'There are no zero-commit months and no gaps of at least three months. The '
                           'declining classification means a lower monthly average, not a stopped project. '
                           'Feature and bug-fix PRs continued through 2025-09-29, so it was Active at the '
                           'cutoff.',
                  'reflection': 'CCL has commits every month in the provided timeline, so there is no '
                                'before/after gap to investigate. The declining label describes the change '
                                'in average commit volume; it does not mean development ended. Recent '
                                'commits still introduce scientific features and fix calculation problems, '
                                'with several contributors involved. The baryonified Duffy change merged in '
                                'September 2025 is a clear example of continued development, so inactivity '
                                'and recovery explanations are not applicable here.',
                  'check': 'There is no gap window for README/issue comparison. Recent PR #1254 adds '
                           'scientific functionality and PR #1248 fixes a normalization problem, supporting '
                           'continued activity.',
                  'sources': [('Recent scientific feature', 'https://github.com/LSSTDESC/CCL/pull/1254'),
                              ('Recent calculation fix', 'https://github.com/LSSTDESC/CCL/pull/1248')]},
 'dsavransky_exosims': {'reason': 'Likely a short coding lull, not abandonment. Although November-December '
                                  '2016 have no commits, maintainer dsavransky replied to issue #13 on '
                                  '2016-11-04, showing support continued: '
                                  'https://github.com/dsavransky/EXOSIMS/issues/13',
                        'recovered': 'Yes; commits resumed on 2017-01-19 (f83b5c071d).',
                        'why': 'January 2017 commits fix simulation and ephemeris problems. Issue #13 was '
                               'diagnosed and closed on 2017-01-20 alongside dda11ab493, an Observatory '
                               'static-ephemerides fix: https://github.com/dsavransky/EXOSIMS/issues/13',
                        'who': 'Existing contributors Christian Delacroix and Dmitry Savransky drive the '
                               'immediate return (f83b5c071d and e789c6d0df). New recorded author jmassarani '
                               'appears in the tenth post-gap commit (d10a51e778), after the return had '
                               'started.',
                        'notes': 'There are no gaps of at least three months; the longest is two months. A '
                                 'maintainer replied to a bug report during that pause, and fixes resumed in '
                                 'January 2017, so a commit gap is not the same as abandonment. The last '
                                 'default-branch commit by the cutoff was 2025-09-30, and recent bot '
                                 'maintenance plus human fixes support Active status.',
                        'reflection': 'This two-month pause is easier to interpret once issue activity is '
                                      'included. The maintainer was still replying in November, so there is '
                                      'evidence against a full project shutdown. January commits fix several '
                                      'simulation problems, and issue #13 is resolved with an ephemerides '
                                      'fix around the same time. Existing contributors lead the return, with '
                                      'one new author appearing later in the sample. Recent automated '
                                      'maintenance is common, but human bug fixes also continue, so activity '
                                      'is not just bots.',
                        'check': 'Issue #13 has a maintainer reply during the gap and a confirmed fix on '
                                 '2017-01-20. The boundary README still lists contributors, funding and '
                                 'documentation, with no shutdown notice; no PRs were found in the searched '
                                 'gap window.',
                        'sources': [('Issue discussion during and after gap',
                                     'https://github.com/dsavransky/EXOSIMS/issues/13'),
                                    ('Ephemerides fix',
                                     'https://github.com/dsavransky/EXOSIMS/commit/dda11ab493')]}}

## Save statistics and reflections

In [13]:
for index, row in stats.iterrows():
    project = row['Project']
    review = reviews[project]
    latest = pd.to_datetime(recent[project]['date'], utc=True).max()
    status = 'Active' if latest >= pd.Timestamp('2025-04-01', tz='UTC') else 'Inactive'
    for column, key in [('HypothesizedGapReason', 'reason'), ('HasRecovered', 'recovered'),
                        ('WhyRecovered', 'why'), ('WhoRecovered', 'who'), ('Notes', 'notes')]:
        stats.loc[index, column] = review[key]
    stats.loc[index, 'Currentstatus'] = status
    row = stats.loc[index]
    gap = (f"{row['LongestGapStart']} to {row['LongestGapEnd']} ({row['LongestGapLength']} months)"
           if row['LongestGapLength'] else 'None')
    links = '\n'.join(f'- [{title}]({url})' for title, url in review['sources'])
    for side in ['pre', 'post']:
        sample = gap_samples[project][side]
        if not sample.empty:
            commit = sample.iloc[-1] if side == 'pre' else sample.iloc[0]
            url = f"https://github.com/{repos[project]}/commit/{commit['commit_sha1']}"
            links += f"\n- [{side}: {commit['commit_sha1'][:10]}]({url})"
    latest_sha = recent[project].loc[pd.to_datetime(recent[project]['date'], utc=True).idxmax(), 'sha']
    links += f"\n- [Latest default-branch commit at cutoff](https://github.com/{repos[project]}/commit/{latest_sha})"
    text = f"""# {project}

Pattern: {row['ActivityPattern']}. Gaps of at least three months: {row['NumberOfGapsInTimeline']}. Longest gap: {gap}.

{review['reflection']}

Before themes: {row['BeforeThemes']}. After themes: {row['AfterThemes']}. Recent themes: {row['RecentThemes']}.

GitHub check: {review['check']}

Status as of 2025-09-30: **{status}**. Last default-branch commit by then: {latest.strftime('%Y-%m-%d')}. The original Part 1 lastGHCommitDate is a later snapshot.

Sources:

{links}
"""
    (root / f'{netid}_reflection_{project}.md').write_text(text, encoding='utf-8')

final_columns = ['Project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks',
                 'lastGHCommitDate', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength',
                 'NumCommitsAfterLastGap', 'ActivityPattern', 'NumberOfGapsInTimeline',
                 'BeforeThemes', 'AfterThemes', 'HypothesizedGapReason', 'HasRecovered',
                 'WhyRecovered', 'WhoRecovered', 'Currentstatus', 'RecentThemes', 'Notes']
stats = stats[final_columns]
stats.to_csv(root / f'{netid}_project_stats.csv', sep=';', index=False)
display(stats[['Project', 'HasRecovered', 'Currentstatus', 'RecentThemes']])
print('Saved gap commits, complete statistics, and 10 reflections.')

,Project,HasRecovered,Currentstatus,RecentThemes
0,vibansal_hapcut2,"Yes, limited; CSV activity resumed in August 2...",Inactive,Documentation updates:Feature development
1,friendly_heplots,Yes; commits resumed on 2008-11-04 (f9c5f2e2af).,Active,Bug fixes:Documentation updates
2,elki-project_elki,Yes; author-dated commits resumed on 2024-02-17.,Active,Other:Feature development
3,neuroml_neuroml2,"Yes; work resumed in April 2024, but recovery ...",Inactive,Release:Documentation updates
4,ropensci_gsodr,Yes; commits resumed on 2016-01-27 (def440c2d4).,Active,Release:Documentation updates
5,modelica_modelica_linearsystems2,Yes; work resumed on 2023-06-15 (cf486a8404).,Active,Documentation updates:Other
6,wangbolun300_fast-envelope,Yes; a new contribution appeared on 2025-03-28...,Active,Other:Documentation updates
7,antvis_g6,"Yes; release work resumed in July 2017, follow...",Active,Documentation updates:Bug fixes
8,lsstdesc_ccl,N/A; no zero-commit month in the observed time...,Active,Bug fixes:Feature development
9,dsavransky_exosims,Yes; commits resumed on 2017-01-19 (f83b5c071d).,Active,Automated bot contributions:Bug fixes


Saved gap commits, complete statistics, and 10 reflections.
